# ResStock load tables for all scenarios

**Purpose:** build the 15-min / hourly / annual ResStock load tables for the optimization from the simulation results of several scenarios (same logic as section 4 of notebook 1, which handles one upgrade at a time). Every scenario is read from its own model folder and written to `data/parquet/` with the scenario name in the file name. Each row carries the AMI meters matched to its building (`meter_ids`, from `results/match_report.csv`).

## Scenarios
Defined in the configuration cell (`SCENARIOS`); the folders are written by notebook 4 (`4_resstock_models_df_run`).

| Scenario name | Models folder | Meaning |
|---|---|---|
| `baseline_rebuilt` | `resstock_models_rebuilt` | baseline, rebuilt with the 2025_Release_1 measures, no flex |
| `up04_rebuilt` | `resstock_models_up04_rebuilt` | upgrade 4 (heat pump), rebuilt, no flex (control for the DF runs) |
| `up04_df_2F` | `resstock_models_up04_df` | upgrade 4 + upgrade 24 (HVAC demand flexibility, 2 F on-peak shed) |

Add entries for other DF variants. Compare scenarios only within the rebuilt runs (see notebook 4); the original runs keep their tables from notebook 1 (`resstock_timeseries_*[_up04]`).

## Workflow overview
| # | Section | Output |
|---|---------|--------|
| 1 | Configuration (scenarios, outputs) | - |
| 2 | Building → meter map (reverse of `results/match_report.csv`) | `results/building_to_meter_map.csv` |
| 3 | Helper functions | - |
| 4 | Build the load tables for each scenario (parallel read per building) | `data/parquet/resstock_timeseries_all_15_<scenario>`, `..._all_hour_<scenario>`, `..._annual_<scenario>` (`.csv` and `.parquet`), each with a `meter_ids` column |
| 5 | Summary across scenarios | table in the notebook |

## Notes
- Run top to bottom, from `grid_edge_fy25/notebooks/`. Needs `pyarrow` (in `requirements.txt`) and `results/match_report.csv` (from the external mapping step).
- Only buildings listed in the scenario's building-ID file with a finished simulation (`run/finished.job`, no `run/failed.job`, `run/eplusout.csv`) are included; the others are reported per scenario.
- Same conventions as notebook 1 so the tables are comparable: column `EMS:EMS_ElectricityFacility_kWh [](TimeStep)` (kWh per 15 min), time labels use the prefix `2024-` (the models ran with a 2023 calendar, no Feb 29), existing outputs of the same scenario are overwritten.
- `meter_ids` is the list of all AMI meters matched to the building, as text such as `[6000973858, 6000973916]`; `[]` means the building is not matched to any meter. A building can be matched to several meters, so this is not a 1-to-1 key; parse it with `ast.literal_eval` if you need the list.
- The 15-min CSVs are large (about 1 GB per scenario before the `meter_ids` column, which adds to every row); set `WRITE_CSV = False` if only the Parquet files are used.



## Setup
Assumes the project environment is already created and activated (`python -m pip install -r requirements.txt`). The cell below is the same in every notebook of this project. Run notebooks from `grid_edge_fy25/notebooks/` (all paths are relative to it).


In [1]:
# Common setup (identical in all notebooks of this project)
import ast
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

# Paths (relative to grid_edge_fy25/notebooks/)
current_path = Path(os.getcwd())
data_path = current_path / ".." / "data"
base_files = data_path / "base_files"        # building id lists, measures, weather, workflow
input_files = data_path / "input_files"      # NY metadata, AMI consumption
parquet_folder = data_path / "parquet"       # load tables written for the optimization
results_path = current_path / ".." / "results"
extract_dir = data_path / "resstock_models"  # bldgXXXXXXX folders (models + simulation output)

parquet_folder.mkdir(parents=True, exist_ok=True)


---
## 1. Configuration
- `SCENARIOS`: scenario name -> models folder (under `data/`, written by notebook 4) and building-ID file (under `data/base_files/`, written by notebook 1). The scenario name becomes the suffix of every output file.
- `SCENARIOS_TO_BUILD`: the scenarios processed in this run (default: all). Scenarios whose models folder does not exist yet are skipped with a message.
- `N_WORKERS`: buildings read in parallel. `WRITE_CSV`: also write the CSV versions (Parquet is always written).


In [2]:
# ========================
# CONFIG
# ========================
# scenario name -> models folder (data/) and building-ID list (data/base_files/)
SCENARIOS = {
    "baseline_rebuilt": dict(models="resstock_models_rebuilt",      ids="building_ids_metadata_2024.csv"),
    "up04_rebuilt":     dict(models="resstock_models_up04_rebuilt", ids="building_ids_metadata_2024_up04.csv"),
    "up04_df_2F":       dict(models="resstock_models_up04_df",      ids="building_ids_metadata_2024_up04.csv"),
}
SCENARIOS_TO_BUILD = list(SCENARIOS)      # e.g. ["up04_df_2F"] to build only one

N_WORKERS = 8          # buildings read in parallel
WRITE_CSV = True       # also write the CSV versions (large: about 1 GB per 15-min table)

YEAR_PREFIX = "2024-"  # the models run with a 2023 calendar; labels use 2024 like notebook 1
DATE_COL = "Date/Time"
TS_COL = "EMS:EMS_ElectricityFacility_kWh [](TimeStep)"   # electricity facility, kWh per 15 min


def out_paths(scenario: str) -> dict:
    """Output files of a scenario in data/parquet/ (scenario name as suffix)."""
    names = {"15": f"resstock_timeseries_all_15_{scenario}",
             "hr": f"resstock_timeseries_all_hour_{scenario}",
             "anl": f"resstock_timeseries_annual_{scenario}"}
    return {k: {"csv": parquet_folder / f"{v}.csv", "pq": parquet_folder / f"{v}.parquet"} for k, v in names.items()}


unknown = [s for s in SCENARIOS_TO_BUILD if s not in SCENARIOS]
if unknown:
    raise KeyError(f"Not in SCENARIOS: {unknown}")

for s in SCENARIOS_TO_BUILD:
    cfg = SCENARIOS[s]
    models, ids = (data_path / cfg["models"]).resolve(), (base_files / cfg["ids"]).resolve()
    print(f"{s:18s} models: {models.name} ({'found' if models.exists() else 'MISSING'}), ids: {ids.name} "
          f"({'found' if ids.exists() else 'MISSING'}) -> {out_paths(s)['15']['pq'].name} ...")


baseline_rebuilt   models: resstock_models_rebuilt (found), ids: building_ids_metadata_2024.csv (found) -> resstock_timeseries_all_15_baseline_rebuilt.parquet ...
up04_rebuilt       models: resstock_models_up04_rebuilt (found), ids: building_ids_metadata_2024_up04.csv (found) -> resstock_timeseries_all_15_up04_rebuilt.parquet ...
up04_df_2F         models: resstock_models_up04_df (found), ids: building_ids_metadata_2024_up04.csv (found) -> resstock_timeseries_all_15_up04_df_2F.parquet ...


---
## 2. Building → meter map
Reverses `results/match_report.csv` (one row per AMI meter, `ResStock_ids` = list of matched ResStock buildings) into one row per building with the list of all AMI meters matched to it. A building can be matched to several meters (the same building appears on up to 14), and many buildings have no meter, so the value is a list: `meter_ids` = `[6000973858, 6000973916]`, or `[]` if the building is not matched to any meter. Saved to `results/building_to_meter_map.csv` (matched buildings only: `building_id`, `meter_id_list`, `n_meters`).

The `meter_ids` column (same list as text) is added to every table written in section 4: 15-min, hourly and annual. The same map applies to all scenarios, since they use the same building IDs.

In [3]:
MATCH_REPORT = results_path / "match_report.csv"
BLDG_METER_MAP_CSV = results_path / "building_to_meter_map.csv"

if not MATCH_REPORT.exists():
    raise FileNotFoundError(f"{MATCH_REPORT} not found (AMI meter -> ResStock building match report)")

mr = pd.read_csv(MATCH_REPORT, usecols=["meter id", "ResStock_ids"])
if mr["meter id"].duplicated().any():
    raise ValueError("Duplicate meter ids in the match report")

# building_id -> sorted list of matched meter ids (plain Python ints)
meters_by_bldg = {}
for meter_id, ids in zip(mr["meter id"], mr["ResStock_ids"]):
    bldgs = ast.literal_eval(ids) if isinstance(ids, str) and ids.strip() else []
    for b in bldgs:
        meters_by_bldg.setdefault(int(b), set()).add(int(meter_id))
meters_by_bldg = {b: sorted(ms) for b, ms in sorted(meters_by_bldg.items())}

# text form written to the load tables, e.g. "[6000973858, 6000973916]"; "[]" = building not matched to any meter
METER_IDS_STR = {b: str(ms) for b, ms in meters_by_bldg.items()}
NO_METERS = "[]"

bmap = pd.DataFrame({"building_id": list(meters_by_bldg),
                     "meter_id_list": [str(ms) for ms in meters_by_bldg.values()],
                     "n_meters": [len(ms) for ms in meters_by_bldg.values()]})
bmap.to_csv(BLDG_METER_MAP_CSV, index=False)

n_pairs = int(bmap["n_meters"].sum())
print(f"Meters in match report: {len(mr)} | with >=1 building: {mr['ResStock_ids'].ne('[]').sum()}")
print(f"Matched buildings: {len(bmap)} | meter-building pairs: {n_pairs} "
      f"| buildings with >1 meter: {int((bmap['n_meters'] > 1).sum())} | max meters per building: {int(bmap['n_meters'].max())}")
print("Saved:", BLDG_METER_MAP_CSV)
bmap.head()

Meters in match report: 538 | with >=1 building: 422
Matched buildings: 187 | meter-building pairs: 422 | buildings with >1 meter: 96 | max meters per building: 14
Saved: c:\Users\jxiong\Documents\GitHub\grid_edge_ithaca\grid_edge_fy25\notebooks\..\results\building_to_meter_map.csv


,building_id,meter_id_list,n_meters
0,7612,[6000981167],1
1,13761,"[6000973858, 6000981276, 6000989106, 600099530...",5
2,15812,"[6000989600, 6000995289, 6001001260]",3
3,21336,"[6000981252, 6000998972, 6001001254]",3
4,22641,[6001000212],1


---
## 3. Helper functions
`process_csv` reads one `eplusout.csv` (time and `TS_COL` only) and returns the building's 15-min rows, hourly sums and annual kWh, each with the building's `meter_ids` from section 2. `build_scenario` runs it for all finished buildings of a scenario in parallel and streams the results into the combined CSV / Parquet files, so the full table never has to fit in memory.

A building is used only if `run/eplusout.csv` and `run/finished.job` exist and there is no `run/failed.job`.



In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed

import pyarrow as pa
import pyarrow.parquet as pq

TABLE_SCHEMA = pa.schema([pa.field("bldg_id", pa.int64()),
                          pa.field("time", pa.timestamp("ns")),
                          pa.field("ts_elec", pa.float64()),
                          pa.field("meter_ids", pa.string())])   # matched AMI meters, e.g. "[6000973858]"
COLS_ORDER = ["bldg_id", "time", "ts_elec", "meter_ids"]


def parse_time_col(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip().str.replace(r"\s{2,}", " ", regex=True)
    s = s.str.replace("24:00:00", "00:00:00", regex=False)
    return pd.to_datetime(YEAR_PREFIX + s, format="%Y-%m/%d %H:%M:%S", errors="coerce")


def process_csv(bldg_id: int, csv_path: Path):
    """Returns (bldg_id, df15, df_hr, ann_kwh, status) for one eplusout.csv."""
    try:
        df = pd.read_csv(csv_path, usecols=[DATE_COL, TS_COL])
    except ValueError as e:
        return bldg_id, None, None, 0.0, f"missing_cols: {e}"
    except Exception as e:
        return bldg_id, None, None, 0.0, f"read_fail: {e}"

    df = df.rename(columns={DATE_COL: "time", TS_COL: "ts_elec"})
    df["time"] = parse_time_col(df["time"]).astype("datetime64[ns]")
    df = df.dropna(subset=["time"])
    df["ts_elec"] = pd.to_numeric(df["ts_elec"], errors="coerce").fillna(0.0)

    meter_ids = METER_IDS_STR.get(bldg_id, NO_METERS)   # list of matched AMI meters (section 2)

    df15 = df.loc[:, ["time", "ts_elec"]].copy()
    df15.insert(0, "bldg_id", bldg_id)
    df15["meter_ids"] = meter_ids

    df_hr = df.assign(time=df["time"].dt.floor("h")).groupby("time", as_index=False)["ts_elec"].sum()
    df_hr.insert(0, "bldg_id", bldg_id)
    df_hr["meter_ids"] = meter_ids

    return bldg_id, df15, df_hr, float(df["ts_elec"].sum()), "ok"


def append_csv(df: pd.DataFrame, path: Path):
    df.loc[:, COLS_ORDER].to_csv(path, index=False, mode="a", header=not path.exists())


class StreamParquetWriter:
    """Append rows to a single Parquet file."""
    def __init__(self, path: Path):
        self.path, self.writer = path, None

    def write(self, df: pd.DataFrame):
        tbl = pa.Table.from_pandas(df.loc[:, COLS_ORDER], schema=TABLE_SCHEMA, preserve_index=False)
        if self.writer is None:
            self.writer = pq.ParquetWriter(str(self.path), schema=TABLE_SCHEMA, compression="snappy")
        self.writer.write_table(tbl)

    def close(self):
        if self.writer is not None:
            self.writer.close()


def build_scenario(scenario: str) -> dict:
    cfg = SCENARIOS[scenario]
    models_dir = (data_path / cfg["models"]).resolve()
    ids_csv = (base_files / cfg["ids"]).resolve()
    if not models_dir.exists() or not ids_csv.exists():
        print(f"[{scenario}] skipped: {models_dir if not models_dir.exists() else ids_csv} not found")
        return {"scenario": scenario, "status": "skipped"}

    building_ids = [int(b) for b in pd.read_csv(ids_csv)["bldg_id"]]
    todo, not_done = [], []
    for bid in building_ids:
        run = models_dir / f"bldg{bid:07d}" / "run"
        if (run / "eplusout.csv").exists() and (run / "finished.job").exists() and not (run / "failed.job").exists():
            todo.append((bid, run / "eplusout.csv"))
        else:
            not_done.append(bid)
    print(f"[{scenario}] {len(todo)} of {len(building_ids)} buildings finished in {models_dir.name}"
          + (f"; not finished: {len(not_done)}" if not_done else ""))
    if not todo:
        return {"scenario": scenario, "status": "no finished buildings", "n_expected": len(building_ids)}

    out = out_paths(scenario)
    for kind in out.values():                      # overwrite earlier outputs of this scenario
        for p in kind.values():
            if p.exists():
                p.unlink()

    w15, whr = StreamParquetWriter(out["15"]["pq"]), StreamParquetWriter(out["hr"]["pq"])
    annual, bad = [], []
    try:
        with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
            futs = [pool.submit(process_csv, bid, p) for bid, p in todo]
            for i, fut in enumerate(as_completed(futs), 1):
                bid, df15, df_hr, ann_kwh, status = fut.result()
                if status != "ok":
                    bad.append((bid, status))
                    print(f"  bldg {bid}: {status}")
                    continue
                if WRITE_CSV:
                    append_csv(df15, out["15"]["csv"])
                    append_csv(df_hr, out["hr"]["csv"])
                w15.write(df15)
                whr.write(df_hr)
                annual.append((bid, ann_kwh))
                if i % 100 == 0 or i == len(todo):
                    print(f"  [{scenario}] {i}/{len(todo)} buildings read")
    finally:
        w15.close()
        whr.close()

    anl_df = pd.DataFrame(annual, columns=["bldg_id", "annl_elec"]).sort_values("bldg_id")
    anl_df["meter_ids"] = anl_df["bldg_id"].map(METER_IDS_STR).fillna(NO_METERS)
    if WRITE_CSV:
        anl_df.to_csv(out["anl"]["csv"], index=False)
    pq.write_table(pa.Table.from_pandas(anl_df, preserve_index=False), out["anl"]["pq"])

    n_matched = int((anl_df["meter_ids"] != NO_METERS).sum())
    print(f"[{scenario}] saved {len(anl_df)} buildings ({n_matched} matched to >=1 AMI meter):")
    for k, label in [("15", "15-min"), ("hr", "hourly"), ("anl", "annual")]:
        print(f"    {label:7s} {out[k]['pq'].name}")
    return {"scenario": scenario, "status": "ok", "n_expected": len(building_ids), "n_built": len(anl_df),
            "not_finished": not_done, "unreadable": bad}


---
## 4. Build the load tables
Runs `build_scenario` for each scenario in `SCENARIOS_TO_BUILD`. The report at the end lists, per scenario, how many buildings were built and which were not finished or unreadable (re-run those in notebook 4 first).



In [5]:
reports = [build_scenario(s) for s in SCENARIOS_TO_BUILD]

print("\n=== Report ===")
for r in reports:
    if r["status"] != "ok":
        print(f"{r['scenario']:18s} {r['status']}")
        continue
    print(f"{r['scenario']:18s} built {r['n_built']} of {r['n_expected']} buildings"
          f" | not finished: {len(r['not_finished'])} | unreadable: {len(r['unreadable'])}")
    if r["not_finished"]:
        print(f"    not finished (first 15): {r['not_finished'][:15]}")
    if r["unreadable"]:
        print(f"    unreadable (first 5): {r['unreadable'][:5]}")


[baseline_rebuilt] 766 of 766 buildings finished in resstock_models_rebuilt
  [baseline_rebuilt] 100/766 buildings read
  [baseline_rebuilt] 200/766 buildings read
  [baseline_rebuilt] 300/766 buildings read
  [baseline_rebuilt] 400/766 buildings read
  [baseline_rebuilt] 500/766 buildings read
  [baseline_rebuilt] 600/766 buildings read
  [baseline_rebuilt] 700/766 buildings read
  [baseline_rebuilt] 766/766 buildings read
[baseline_rebuilt] saved 766 buildings (187 matched to >=1 AMI meter):
    15-min  resstock_timeseries_all_15_baseline_rebuilt.parquet
    hourly  resstock_timeseries_all_hour_baseline_rebuilt.parquet
    annual  resstock_timeseries_annual_baseline_rebuilt.parquet
[up04_rebuilt] 414 of 483 buildings finished in resstock_models_up04_rebuilt; not finished: 69
  [up04_rebuilt] 100/414 buildings read
  [up04_rebuilt] 200/414 buildings read
  [up04_rebuilt] 300/414 buildings read
  [up04_rebuilt] 400/414 buildings read
  [up04_rebuilt] 414/414 buildings read
[up04_rebuil

---
## 5. Summary across scenarios
Annual electricity per scenario, read back from the annual Parquet files (kWh per building and year). `n_common` compares scenarios on the buildings present in all built scenarios, so the means are like-for-like.



In [6]:
annual = {}
for s in SCENARIOS:
    p = out_paths(s)["anl"]["pq"]
    if p.exists():
        annual[s] = pd.read_parquet(p).set_index("bldg_id")["annl_elec"]

if annual:
    wide = pd.DataFrame(annual)
    common = wide.dropna()
    summary = pd.DataFrame({
        "n_buildings": wide.count(),
        "mean_kWh": wide.mean().round(0),
        "n_common": len(common),
        "mean_kWh_common": common.mean().round(0),
    })
    display(summary)
else:
    print("No annual tables found in", parquet_folder)


,n_buildings,mean_kWh,n_common,mean_kWh_common
baseline_rebuilt,766,9281.0,414,8549.0
up04_rebuilt,414,16875.0,414,16875.0
up04_df_2F,414,16631.0,414,16631.0
